In [4]:
import numpy as np

import os 
import sys
sys.path.append("..//utils/")
sys.path.append("..//anatomy/")
sys.path.append("..//neural/")
import color_utils, make_data_dict
import get_probe_coords
import format_waveform_data, waveform_analysis, waveform_plots
import format_chronic_stim
import matplotlib.pyplot as plt

In [5]:
''' Set file paths '''
root_dir = "Z:/Isabel/data/hpc_implants/"
data_file = f"{root_dir}stim_session_data.npy"
save_figs = f"../figures/antidromic_hpc_to_lhy/"

In [6]:
''' Params '''
# data params
sampling_rate = 30000
t_pre = 0.02 # seconds collected before stim starts
t_post = 0.03 # seconds collected after stim time
spk_thresh = 25 # minimum spike amplitude in uV

# stim response window
buffer = 6 # samples
start_t = 5e-3 # seconds
end_t = 15e-3 # seconds
start_idx = np.round((t_pre + start_t)*sampling_rate).astype(int) # samples
end_idx = np.round((t_pre + end_t)*sampling_rate).astype(int) # samples

''' Load the data dictionary of all good stim sessions '''
bird_ids = []
data_dict = np.load(data_file, allow_pickle=True).item()
for bird in data_dict.keys():
    bird_ids.append(bird)
print(f'current birds with saved data: {bird_ids}')

current birds with saved data: ['LIM63', 'RBY94', 'AMB154', 'SLV132', 'IND67', 'LMN146']


In [7]:
''' Identify stim sessions '''
stim_sessions = []
ephys_dirs = []
for bird in bird_ids:
    session_list = data_dict[bird]['all_sessions']
    for session_id in session_list:
        # specify the file paths/session params
        if ('stim' in data_dict[bird][session_id]['preprocessed_data']) & ('ephys' in data_dict[bird][session_id]['preprocessed_data']):
            session_dir = f'{root_dir}{bird}/{bird}_{session_id}/'
            for folder in os.listdir(session_dir):
                if f'{bird}_{session_id}' in folder:
                    ephys_id = folder[-13:]
                    ephys_dir = f"{session_dir}{bird}_{ephys_id}/raw_ephys_output/"
                    stim_sessions.append(f'{bird}_{session_id}')
                    ephys_dirs.append(ephys_dir)

In [8]:
bird = 'LIM63'
session_list = data_dict[bird]['all_sessions']
session_id = session_list[1]

In [9]:
ephys_dir = ephys_dirs[stim_sessions.index(f'{bird}_{session_id}')]

# get the stim params
stim_params = []
for file in os.listdir(ephys_dir):
    if ('neg' in file) & ('amplifier' in file):
        file_parts = file.split(sep='_')
        if 'neg' in file_parts[-1]:
            stim_pol = file_parts[-1][:-4]
            stim_params.append(stim_pol)

In [46]:
for idx, stim_pol in enumerate(stim_params):
    print(f'loading stim data for {session_id}_{stim_pol}')
    # load and preprocess the stim responses
    raw_ephys = format_chronic_stim.load_stim(ephys_dir, stim_pol=stim_pol)
    ephys_data, ch_names = format_chronic_stim.sort_stim_by_channel(ephys_dir, raw_ephys)
    [n_channels, n_samples, n_stim] = ephys_data.shape

    # get the stim times
    stim_times = np.load(f'{ephys_dir}stim_t_{stim_pol}.npy')
    stim_times = np.squeeze(stim_times.astype(int))

    # compute the average stim response (hash)
    filt_data = format_chronic_stim.filter_stim_for_spikes(ephys_data)
    stim_hash = np.moveaxis(filt_data[:, start_idx:end_idx], -1, 0)
    avg_hash = np.mean(stim_hash, axis=0)

    # identify channels with an antidromic response (worm)
    if idx == 0:
        worm_ch_idx = np.zeros(n_channels).astype(bool)
    for i in range(n_channels):
        if any(np.abs(avg_hash[i]) >= spk_thresh):
            worm_ch_idx[i] = True
            print(ch_names[i])
print(f'{session_id} has {np.sum(worm_ch_idx)} total channels with stim responses ($p \leq 0.01$)')

<>:24: SyntaxWarning: invalid escape sequence '\l'
<>:24: SyntaxWarning: invalid escape sequence '\l'
C:\Users\Isabel\AppData\Local\Temp\ipykernel_9724\846916953.py:24: SyntaxWarning: invalid escape sequence '\l'
  print(f'{session_id} has {np.sum(worm_ch_idx)} total channels with stim responses ($p \leq 0.01$)')


loading stim data for 240610_neg350
loaded 1500 stim events in 6.493742999999085 seconds
Z:/Isabel/data/hpc_implants/LIM63/LIM63_240610/LIM63_240610_131820/raw_ephys_output/intan_info.mat
A-07-3
A-08-2
A-09-1
A-09-3
A-10-2
A-11-3
A-12-2
A-13-1
A-13-3
A-14-2
A-15-1
A-15-3
A-17-1
A-17-3
A-18-2
A-19-1
A-19-3
A-20-2
A-21-1
A-21-3
A-22-2
B-03-3
B-04-2
B-05-3
B-06-2
B-07-1
B-08-2
B-09-1
B-09-3
B-10-2
B-11-1
B-11-3
B-13-1
B-13-3
B-14-2
B-15-1
B-15-3
B-16-2
B-17-1
B-17-3
B-18-2
B-19-1
B-19-3
B-20-2
B-21-1
B-21-3
B-22-2
loading stim data for 240610_neg150
loaded 665 stim events in 1.3648204999917652 seconds
Z:/Isabel/data/hpc_implants/LIM63/LIM63_240610/LIM63_240610_131820/raw_ephys_output/intan_info.mat
A-07-3
A-08-2
A-09-1
A-09-3
A-10-2
A-11-3
A-12-2
A-13-1
A-13-3
A-14-2
A-15-3
A-17-1
A-17-3
A-18-2
A-19-1
A-19-3
A-20-2
A-21-1
A-21-3
A-22-2
B-09-1
B-10-2
B-11-1
B-11-3
B-13-1
B-13-3
B-14-2
B-15-1
B-15-3
B-16-2
B-17-1
B-17-3
B-18-2
B-19-1
B-21-1
loading stim data for 240610_neg100
loaded 950 sti

In [37]:
print(rf'{session_id} has {np.sum(worm_ch_idx)} total channels with stim responses (p <= 0.01)')

240610 has 47 total channels with stim responses (p <= 0.01)


In [38]:
# check for collision dict files and collect projection cells
# TODO add "keep_idx" to session spreadsheet to exclude false positives
# ...or longer term TODO just improve the collision detection
all_sig_cells = np.asarray([])
for file in sorted(os.listdir(ephys_dir[:-17])):
    if 'collision_props' in file:
        collision_dict = np.load(f'{ephys_dir[:-17]}{file}', allow_pickle=True).item()
        sig_cells = collision_dict['sig_cell_IDs']
        all_sig_cells = np.append(all_sig_cells, sig_cells)

# remove double counted cells
all_sig_cells, unique_idx = np.unique(all_sig_cells, return_index=True)
all_sig_cells = all_sig_cells.astype(int)
print(f'{session_id} has {all_sig_cells.shape[0]} total cells with significant collisions')

FileNotFoundError: [Errno 2] No such file or directory: 'collision_props_neg100.npy'

In [39]:
ch_names

['A-01-2',
 'A-02-2',
 'A-03-1',
 'A-03-3',
 'A-04-2',
 'A-05-1',
 'A-05-3',
 'A-06-2',
 'A-07-1',
 'A-07-3',
 'A-08-2',
 'A-09-1',
 'A-09-3',
 'A-10-2',
 'A-11-1',
 'A-11-3',
 'A-12-2',
 'A-13-1',
 'A-13-3',
 'A-14-2',
 'A-15-1',
 'A-15-3',
 'A-16-2',
 'A-17-1',
 'A-17-3',
 'A-18-2',
 'A-19-1',
 'A-19-3',
 'A-20-2',
 'A-21-1',
 'A-21-3',
 'A-22-2',
 'B-01-2',
 'B-02-2',
 'B-03-1',
 'B-03-3',
 'B-04-2',
 'B-05-1',
 'B-05-3',
 'B-06-2',
 'B-07-1',
 'B-07-3',
 'B-08-2',
 'B-09-1',
 'B-09-3',
 'B-10-2',
 'B-11-1',
 'B-11-3',
 'B-12-2',
 'B-13-1',
 'B-13-3',
 'B-14-2',
 'B-15-1',
 'B-15-3',
 'B-16-2',
 'B-17-1',
 'B-17-3',
 'B-18-2',
 'B-19-1',
 'B-19-3',
 'B-20-2',
 'B-21-1',
 'B-21-3',
 'B-22-2']

In [41]:
ks_dir = "kilosort4_blanked/"

In [44]:
ch_pos_probe = np.load(f"{ephys_dir[:-17]}{ks_dir}channel_positions.npy")

In [54]:
ch_pos_probe

array([[ 18.5,   0. ],
       [ 18.5,  30. ],
       [  0. ,  45. ],
       [ 37. ,  45. ],
       [ 18.5,  60. ],
       [  0. ,  75. ],
       [ 37. ,  75. ],
       [ 18.5,  90. ],
       [  0. , 105. ],
       [ 37. , 105. ],
       [ 18.5, 120. ],
       [  0. , 135. ],
       [ 37. , 135. ],
       [ 18.5, 150. ],
       [  0. , 165. ],
       [ 37. , 165. ],
       [ 18.5, 180. ],
       [  0. , 195. ],
       [ 37. , 195. ],
       [ 18.5, 210. ],
       [  0. , 225. ],
       [ 37. , 225. ],
       [ 18.5, 240. ],
       [  0. , 255. ],
       [ 37. , 255. ],
       [ 18.5, 270. ],
       [  0. , 285. ],
       [ 37. , 285. ],
       [ 18.5, 300. ],
       [  0. , 315. ],
       [ 37. , 315. ],
       [ 18.5, 330. ],
       [168.5,   0. ],
       [168.5,  30. ],
       [150. ,  45. ],
       [187. ,  45. ],
       [168.5,  60. ],
       [150. ,  75. ],
       [187. ,  75. ],
       [168.5,  90. ],
       [150. , 105. ],
       [187. , 105. ],
       [168.5, 120. ],
       [150

In [61]:
shank_dist = 150
ml_A = 200
ml_B = 140

In [70]:
# get the two angles
alpha = np.deg2rad(135)
delta = np.deg2rad(45)

# get the terms for the distance equation
Acos = ml_A * np.cos(delta)
Asin = ml_A * np.sin(delta)
Bcos = ml_B * np.cos(alpha)
Bsin = ml_B * np.sin(alpha)

# get the distance along the DM/DL boundary line
dist_dmdl = Bcos + Acos + np.sqrt(shank_dist**2 - (Bsin - Asin)**2)

# this is the hypotenuse of a 45/45/90 triangle with the AP distance
ap_dist = dist_dmdl / np.sqrt(2)

In [71]:
ap_dist

131.73494974687904

In [72]:
foo = np.asarray([0, 1, 2, 3])

In [78]:
bar = foo.astype(bool)
bar.dtype

dtype('bool')